In [0]:
# ==========================================================
# Project      : Social Media Sentiment Analysis
# Notebook     : test_silver
# Description  : Silver Layer Testing
# ==========================================================

tables = [

    "silver_valid_tweets",
    "silver_tweets",
    "silver_sentiment",
    "silver_trends",
    "silver_user_metadata"

]

In [0]:
# ==========================================================
# Test 1 : Silver Tables Exist
# ==========================================================

for table in tables:

    assert spark.catalog.tableExists(
        f"socialmedia_databricks.silver.{table}"
    ), f"{table} does not exist"

print("✅ Test 1 Passed : All Silver tables exist")

✅ Test 1 Passed : All Silver tables exist


In [0]:
# ==========================================================
# Test 2 : Row Count Validation
# ==========================================================

for table in tables:

    row_count = spark.table(
        f"socialmedia_databricks.silver.{table}"
    ).count()

    assert row_count > 0, f"{table} is empty"

print("✅ Test 2 Passed : All Silver tables contain data")

✅ Test 2 Passed : All Silver tables contain data


In [0]:
# ==========================================================
# Test 3 : Required Columns Validation
# ==========================================================

required_columns = [

    "tweet_id",
    "topic_category",
    "tweet_text",
    "tweet_timestamp",
    "impressions",
    "likes",
    "retweets",
    "replies",
    "engagement_count",
    "sentiment_score"

]

df = spark.table("socialmedia_databricks.silver.silver_valid_tweets")

for column in required_columns:

    assert column in df.columns, f"{column} missing"

print("✅ Test 3 Passed : Required columns validated")

✅ Test 3 Passed : Required columns validated


In [0]:
# ==========================================================
# Test 4 : Null Validation
# ==========================================================

from pyspark.sql.functions import col

df = spark.table("socialmedia_databricks.silver.silver_valid_tweets")

critical_columns = [

    "tweet_id",
    "tweet_text",
    "tweet_timestamp"

]

for column in critical_columns:

    null_count = df.filter(col(column).isNull()).count()

    assert null_count == 0, f"{column} contains NULL values"

print("✅ Test 4 Passed : No NULL values found")

✅ Test 4 Passed : No NULL values found


In [0]:
# ==========================================================
# Test 5 : Duplicate Validation
# ==========================================================

from pyspark.sql.functions import col

df = spark.table("socialmedia_databricks.silver.silver_valid_tweets")

duplicate_count = (
    df.groupBy("tweet_id")
      .count()
      .filter(col("count") > 1)
      .count()
)

assert duplicate_count == 0, \
    f"Found {duplicate_count} duplicate tweet_id records"

print("✅ Test 5 Passed : No duplicate tweet_id found")

✅ Test 5 Passed : No duplicate tweet_id found


In [0]:
# ==========================================================
# Test 6 : Metadata Columns Validation
# ==========================================================

metadata_columns = [

    "silver_load_time",
    "pipeline_name",
    "source_system",
    "ingestion_date"

]

for table in tables:

    df = spark.table(f"socialmedia_databricks.silver.{table}")

    for column in metadata_columns:

        assert column in df.columns, f"{column} missing in {table}"

print("✅ Test 6 Passed : Metadata columns validated")

✅ Test 6 Passed : Metadata columns validated


In [0]:
# ==========================================================
# Test 7 : Metadata Values Validation
# ==========================================================

from pyspark.sql.functions import col

df = spark.table("socialmedia_databricks.silver.silver_valid_tweets")

assert df.filter(col("pipeline_name").isNull()).count() == 0
assert df.filter(col("source_system").isNull()).count() == 0
assert df.filter(col("ingestion_date").isNull()).count() == 0

print("✅ Test 7 Passed : Metadata values validated")

✅ Test 7 Passed : Metadata values validated


In [0]:
# ==========================================================
# Test 8 : Data Type Validation
# ==========================================================

df = spark.table("socialmedia_databricks.silver.silver_valid_tweets")

expected_datatypes = {

    "tweet_id": "string",
    "topic_category": "string",
    "tweet_text": "string",
    "tweet_timestamp": "timestamp",
    "impressions": "int",
    "likes": "int",
    "retweets": "int",
    "replies": "int",
    "engagement_count": "int",
    "sentiment_score": "double",
    "bronze_load_time": "timestamp",
    "pipeline_name": "string",
    "source_system": "string",
    "ingestion_date": "date",
    "tweet_date": "date",
    "silver_load_time": "timestamp"

}

actual_datatypes = dict(df.dtypes)

for column, datatype in expected_datatypes.items():

    assert actual_datatypes[column] == datatype, \
        f"{column} should be {datatype}, found {actual_datatypes[column]}"

print("✅ Test 8 Passed : Data types validated")

✅ Test 8 Passed : Data types validated


In [0]:
spark.table("socialmedia_databricks.silver.silver_valid_tweets").printSchema()

root
 |-- tweet_id: string (nullable = true)
 |-- topic_category: string (nullable = true)
 |-- tweet_text: string (nullable = true)
 |-- tweet_timestamp: timestamp (nullable = true)
 |-- impressions: integer (nullable = true)
 |-- likes: integer (nullable = true)
 |-- retweets: integer (nullable = true)
 |-- replies: integer (nullable = true)
 |-- engagement_count: integer (nullable = true)
 |-- sentiment_score: double (nullable = true)
 |-- bronze_load_time: timestamp (nullable = true)
 |-- pipeline_name: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- tweet_date: date (nullable = true)
 |-- silver_load_time: timestamp (nullable = true)

